# 02. 데이터 전처리 — 전후 비교

`data/raw_sample.csv`를 정제하고 전/후를 비교한다. 산출물: `data/{train,val,test}.csv`

**정제 규칙** (`src.data.pipeline`):

| 단계 | 내용 | 이유 |
|---|---|---|
| 기본 정제 | 결측·빈문자열·중복 제거, 라벨 1/-1 → 1/0 | 학습 불가능한 행 제거 |
| BBCode 제거 | `[h1]`, `[b]`, `[spoiler]` 등 | Steam 마크업은 감성과 무관 |
| URL 제거 | `http://...`, `www...` | 어휘 낭비 |
| 반복문자 축약 | `soooooo`→`soo`, `♥♥♥♥`→`♥♥` | 같은 의미 토큰 통합 |
| 구두점 분리 | `game.`→`game .` | LSTM 공백 토크나이저의 어휘 낭비 방지 |
| 최소 길이 필터 | 3단어 미만 제거 | `"gud"`, `"+"` 같은 무신호 리뷰 제거 |

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import DATA_DIR, RANDOM_SEED
from src.data.pipeline import clean_reviews, split_data, normalize_text
from src.models.dataset import build_vocab

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

raw = pd.read_csv(DATA_DIR / "raw_sample.csv")
print(f"원본: {len(raw):,}건")

## 전/후 예시 — 노이즈가 실제로 어떻게 정리되는가

In [ ]:
texts = raw["review_text"].astype(str)
noisy_mask = (texts.str.contains(r"\[/?[a-zA-Z0-9*=]+\]|https?://|(.)\2{3,}", regex=True)
              & (texts.str.split().str.len() < 60))
examples = texts[noisy_mask].head(8)

pd.set_option("display.max_colwidth", 120)
compare = pd.DataFrame({"전 (원본)": examples.values,
                        "후 (정제)": [normalize_text(t) for t in examples]})
compare

## 전/후 통계 비교 — 데이터 크기, 어휘 크기, 평균 길이

In [ ]:
before = clean_reviews(raw, "review_text", "review_score")          # 기본 정제만
after = clean_reviews(raw, "review_text", "review_score",
                      normalize=True, min_words=3)                  # 정규화 + 길이 필터

stats = pd.DataFrame({
    "리뷰 수": [len(before), len(after)],
    "어휘 크기 (min_freq=2)": [len(build_vocab(before["text"].tolist())),
                               len(build_vocab(after["text"].tolist()))],
    "평균 단어 수": [before["text"].str.split().str.len().mean(),
                     after["text"].str.split().str.len().mean()],
    "긍정 비율": [before["label"].mean(), after["label"].mean()],
}, index=["전처리 전", "전처리 후"]).round(3)
stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ["리뷰 수", "어휘 크기 (min_freq=2)", "평균 단어 수"]):
    vals = stats[col]
    ax.bar(vals.index, vals.values, color=["#8c8c8c", "#4c72b0"])
    ax.set_title(col)
    for i, v in enumerate(vals.values):
        ax.text(i, v, f"{v:,.1f}" if v % 1 else f"{int(v):,}", ha="center", va="bottom")
plt.suptitle("전처리 전/후 비교")
plt.tight_layout()
plt.show()

In [ ]:
# 리뷰 길이 분포 전/후 오버레이
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(before["text"].str.split().str.len().clip(upper=200), bins=50,
        alpha=0.5, label="전처리 전", color="#8c8c8c")
ax.hist(after["text"].str.split().str.len().clip(upper=200), bins=50,
        alpha=0.5, label="전처리 후", color="#4c72b0")
ax.set_title("리뷰 길이 분포 전/후 (단어 수, 200+ 절단)")
ax.set_xlabel("단어 수")
ax.set_ylabel("리뷰 수")
ax.legend()
plt.tight_layout()
plt.show()

## 감성별 상위 토큰 — 정제 후 어떤 단어가 신호가 되는가

In [ ]:
from collections import Counter

STOP = set("the a an and or but of to in is it this that i you for on with was are be "
           "my as at so if have has not can its . , ! ?".split())

def top_tokens(texts, n=20):
    c = Counter(w for t in texts for w in t.lower().split() if w not in STOP and len(w) > 1)
    return c.most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (lab, name, color) in zip(axes, [(1, "긍정", "#4c72b0"), (0, "부정", "#c44e52")]):
    toks = top_tokens(after[after["label"] == lab]["text"])
    words, freqs = zip(*toks)
    ax.barh(range(len(words)), freqs, color=color)
    ax.set_yticks(range(len(words)), words)
    ax.invert_yaxis()
    ax.set_title(f"{name} 리뷰 상위 토큰")
plt.tight_layout()
plt.show()

## 분할 및 저장

In [ ]:
tr, va, te = split_data(after, seed=RANDOM_SEED)
tr.to_csv(DATA_DIR / "train.csv", index=False)
va.to_csv(DATA_DIR / "val.csv", index=False)
te.to_csv(DATA_DIR / "test.csv", index=False)
print(f"train={len(tr)} val={len(va)} test={len(te)} → {DATA_DIR}")

In [ ]:
# 분할별 크기와 라벨 비율 (계층분할 검증)
splits = {"train": tr, "val": va, "test": te}
fig, ax = plt.subplots(figsize=(7, 4))
pos = [len(d[d["label"] == 1]) for d in splits.values()]
neg = [len(d[d["label"] == 0]) for d in splits.values()]
ax.bar(splits.keys(), pos, label="긍정", color="#4c72b0")
ax.bar(splits.keys(), neg, bottom=pos, label="부정", color="#c44e52")
for i, d in enumerate(splits.values()):
    ax.text(i, len(d), f"긍정 {d['label'].mean():.1%}", ha="center", va="bottom")
ax.set_title("train/val/test 분할 크기와 라벨 비율")
ax.set_ylabel("리뷰 수")
ax.legend()
plt.tight_layout()
plt.show()